In [ ]:
import os
import boto3
from dotenv import load_dotenv

# Load local secrets from .env; never store access keys in notebook source.
load_dotenv()
R2_ACCESS_KEY_ID = os.environ["R2_ACCESS_KEY_ID"]
R2_SECRET_ACCESS_KEY = os.environ["R2_SECRET_ACCESS_KEY"]
R2_ENDPOINT_URL = os.environ["R2_ENDPOINT_URL"]
BUCKET_NAME = os.environ["R2_BUCKET_NAME"]

s3_client = boto3.client(
    "s3",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    endpoint_url=R2_ENDPOINT_URL,
    region_name=os.getenv("R2_REGION", "auto"),
)
print("R2 client configured from environment variables; no object was uploaded.")


In [4]:
import json
import os

# ==========================================
# 1. HÀM ĐỌC NỘI DUNG MỘT FILE VĂN BẢN / TEXT CỤ THỂ
# ==========================================
def read_text_file_from_r2(file_key: str):
  try:
    response = s3_client.get_object(Bucket=BUCKET_NAME, Key=file_key)
    # Đọc nội dung file dưới dạng chuỗi (string)
    content = response["Body"].read().decode("utf-8")
    print(f"📖 [ĐỌC FILE THÀNH CÔNG]: {file_key}")
    print(f"--- Nội dung -- -\n{content}\n----------------")
    return content
  except Exception as e:
    print(f"❌ Lỗi khi đọc file {file_key}: {e}")
    return None


# ==========================================
# 2. HÀM LIỆT KÊ TẤT CẢ CÁC FILE TRONG DATA LAKE
# ==========================================
def list_files_in_bronze_lake(prefix="bronze_lake/"):
  print(f"🔍 Đang quét danh sách file trong bucket với tiền tố: '{prefix}'...")
  response = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix)

  if "Contents" in response:
    print(f"📂 Tìm thấy các đối tượng sau:")
    for obj in response["Contents"]:
      print(f" - File: {obj['Key']} ({obj['Size']} bytes)")
  else:
    print("⚠️ Không tìm thấy file nào trong thư mục này.")


# ==========================================
# 3. HÀM ĐỌC VÀ PARSE DỮ LIỆU JSON LỰA CHỌN TỪ R2 (Cho tầng Silver)
# ==========================================
def read_jsonl_from_r2(file_key: str):
  try:
    response = s3_client.get_object(Bucket=BUCKET_NAME, Key=file_key)
    lines = response["Body"].read().decode("utf-8").splitlines()

    data_records = []
    for line in lines:
      if line.strip():
        data_records.append(json.loads(line))

    print(f"🔄 Đã đọc và parse thành công {len(data_records)} bản ghi từ {file_key}")
    return data_records
  except Exception as e:
    print(f"❌ Lỗi khi đọc file JSONL: {e}")
    return []


# ==========================================
# THỰC THI KIỂM TRA THỰC TẾ
# ==========================================
if __name__ == "__main__":
  # 1. Liệt kê toàn bộ file có trên R2
  list_files_in_bronze_lake(prefix="")

  # 2. Đọc thử file test connection bạn vừa upload lên
  read_text_file_from_r2("test_connection.txt")

🔍 Đang quét danh sách file trong bucket với tiền tố: ''...
📂 Tìm thấy các đối tượng sau:
 - File: test_connection.txt (72 bytes)
📖 [ĐỌC FILE THÀNH CÔNG]: test_connection.txt
--- Nội dung ---
Hello Cloudflare R2! Kết nối tầng Bronze (Data Lake) thành công.
----------------


In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Use the same local PostgreSQL configuration as the pipeline.
load_dotenv()
engine = create_engine(os.environ["POSTGRES_DSN"])

query = """
    SELECT source, list_id, payload_format, object_key, indexed_at
    FROM ops.bronze_payload_index
    ORDER BY indexed_at DESC
    LIMIT 10;
"""

df = pd.read_sql(query, con=engine)
print("Recent Bronze payload index entries:")
print(df)
